# 01 — Data Loading & Inspection
**Project:** Telco Customer Churn Prediction  
**Author:** Udit Jadli  
**Dataset:** Maven Analytics — Telecom Customer Churn  

## Objective
Load the raw CSV files, understand the structure of the data, 
check for missing values, and prepare it for SQL loading and EDA.

## 1. Imports

We will import the necessary libraries required for this project.

In [1]:
import pandas as pd      # main library for working with tables/dataframes
import numpy as np       # numerical operations, pandas is built on this
import os                # for building file paths that work on any machine
import warnings

warnings.filterwarnings('ignore')  # hides minor warnings that clutter output

print("Libraries imported successfully!")

Libraries imported successfully!


## 2. File Paths

We define the file paths once here so they can be reused throughout the notebook without hardcoding them repeatedly.

In [2]:
# go one level up from notebooks/ to reach the project root, then into data/raw/
DATA_RAW = os.path.join('..', 'data', 'raw')

# define each file path
CHURN_FILE  = os.path.join(DATA_RAW, 'telecom_customer_churn.csv')
ZIPCODE_FILE = os.path.join(DATA_RAW, 'telecom_zipcode_population.csv')
DICT_FILE   = os.path.join(DATA_RAW, 'telecom_data_dictionary.csv')

# confirm the paths look correct
print("Churn file:   ", CHURN_FILE)
print("Zipcode file: ", ZIPCODE_FILE)
print("Dict file:    ", DICT_FILE)

Churn file:    ..\data\raw\telecom_customer_churn.csv
Zipcode file:  ..\data\raw\telecom_zipcode_population.csv
Dict file:     ..\data\raw\telecom_data_dictionary.csv


## 3. Load Data
We load all three CSV files into pandas DataFrames, which will be used for inspection and analysis throughout this notebook.

In [3]:
# load all three files into dataframes
df_churn = pd.read_csv(CHURN_FILE, encoding='latin-1')
df_zip   = pd.read_csv(ZIPCODE_FILE, encoding='latin-1')
df_dict  = pd.read_csv(DICT_FILE, encoding='latin-1')

# quick confirmation of what we loaded
print(f"Churn table:      {df_churn.shape[0]:,} rows × {df_churn.shape[1]} columns")
print(f"Zip code table:   {df_zip.shape[0]:,} rows × {df_zip.shape[1]} columns")
print(f"Data dictionary:  {df_dict.shape[0]:,} rows × {df_dict.shape[1]} columns")

Churn table:      7,043 rows × 38 columns
Zip code table:   1,671 rows × 2 columns
Data dictionary:  40 rows × 3 columns


## 4. Column Names
We inspect all 38 column names in the main churn table to understand what information is available before diving into the data.

In [4]:
# see all column names in the main churn table
# .tolist() converts the index to a plain Python list — easier to read
print("Columns in churn table:\n")
for col in df_churn.columns.tolist():
    print(" ", col)

Columns in churn table:

  Customer ID
  Gender
  Age
  Married
  Number of Dependents
  City
  Zip Code
  Latitude
  Longitude
  Number of Referrals
  Tenure in Months
  Offer
  Phone Service
  Avg Monthly Long Distance Charges
  Multiple Lines
  Internet Service
  Internet Type
  Avg Monthly GB Download
  Online Security
  Online Backup
  Device Protection Plan
  Premium Tech Support
  Streaming TV
  Streaming Movies
  Streaming Music
  Unlimited Data
  Contract
  Paperless Billing
  Payment Method
  Monthly Charge
  Total Charges
  Total Refunds
  Total Extra Data Charges
  Total Long Distance Charges
  Total Revenue
  Customer Status
  Churn Category
  Churn Reason


## 5. Data Preview
We take a quick look at the first few rows of the dataset to get a feel for the data and spot anything unusual early on.

In [5]:
# .head() shows the first 5 rows by default
# .T transposes it (flips rows and columns) so it's easier to read
# without .T, 38 columns would be squashed horizontally
df_churn.head(3).T

,0,1,2
Customer ID,0002-ORFBO,0003-MKNFE,0004-TLHLJ
Gender,Female,Male,Male
Age,37,46,50
Married,Yes,No,No
Number of Dependents,0,0,0
City,Frazier Park,Glendale,Costa Mesa
Zip Code,93225,91206,92627
Latitude,34.827662,34.162515,33.645672
Longitude,-118.999073,-118.203869,-117.922613
Number of Referrals,2,0,0


## 6. Data Types
We check the data type assigned to each column to ensure pandas has interpreted them correctly, as incorrect types can cause silent errors downstream.

In [6]:
# dtypes tells us what type pandas assigned to each column
# object = text/string, int64 = whole number, float64 = decimal number
print("Column data types:\n")
print(df_churn.dtypes)

Column data types:

Customer ID                           object
Gender                                object
Age                                    int64
Married                               object
Number of Dependents                   int64
City                                  object
Zip Code                               int64
Latitude                             float64
Longitude                            float64
Number of Referrals                    int64
Tenure in Months                       int64
Offer                                 object
Phone Service                         object
Avg Monthly Long Distance Charges    float64
Multiple Lines                        object
Internet Service                      object
Internet Type                         object
Avg Monthly GB Download              float64
Online Security                       object
Online Backup                         object
Device Protection Plan                object
Premium Tech Support               

## 7. Missing Values
We identify columns with null values and understand why they are missing — distinguishing between data quality issues and expected business logic.

In [7]:
# count nulls per column and calculate the percentage
# this tells us both how many and how severe the missingness is
null_count = df_churn.isnull().sum()
null_pct   = (null_count / len(df_churn) * 100).round(1)

# combine into one clean table, only show columns that actually have nulls
null_summary = pd.DataFrame({
    'Missing Count': null_count,
    'Missing %': null_pct
}).query('`Missing Count` > 0').sort_values('Missing Count', ascending=False)

print(f"Columns with missing values: {len(null_summary)} out of {df_churn.shape[1]}\n")
print(null_summary)

Columns with missing values: 15 out of 38



                                   Missing Count  Missing %
Churn Category                              5174       73.5
Churn Reason                                5174       73.5
Offer                                       3877       55.0
Internet Type                               1526       21.7
Avg Monthly GB Download                     1526       21.7
Online Security                             1526       21.7
Online Backup                               1526       21.7
Device Protection Plan                      1526       21.7
Premium Tech Support                        1526       21.7
Streaming TV                                1526       21.7
Streaming Movies                            1526       21.7
Streaming Music                             1526       21.7
Unlimited Data                              1526       21.7
Avg Monthly Long Distance Charges            682        9.7
Multiple Lines                               682        9.7


## Missing Value Findings

- **Churn Category & Reason (73.5%)** — only populated for churned customers. Expected, not an error.
- **Internet columns (21.7%)** — customers with no internet service. Will fill with 'No Internet Service'.
- **Offer (55%)** — no marketing offer accepted. Will fill with 'No Offer'.
- **Long distance columns (9.7%)** — customers with no phone service. Will fill with 0.

All missing values are **structural** — they reflect the business logic, not data quality issues.

## 8. Target Variable
We examine the distribution of our target variable `Customer Status` and create a binary `Churn` column that our ML models will predict.

In [8]:
# check the distribution of our target variable
status_counts = df_churn['Customer Status'].value_counts()
status_pct    = (status_counts / len(df_churn) * 100).round(1)

print("Customer Status distribution:\n")
for status, count in status_counts.items():
    print(f"  {status:<10}  {count:>5,}  ({status_pct[status]}%)")

Customer Status distribution:

  Stayed      4,720  (67.0%)
  Churned     1,869  (26.5%)
  Joined        454  (6.4%)


In [9]:
# create binary churn column — this is what our ML models will predict
# 1 = churned, 0 = stayed or joined (both are active customers)
df_churn['Churn'] = (df_churn['Customer Status'] == 'Churned').astype(int)

# verify it looks right
print("Churn column distribution:\n")
print(df_churn['Churn'].value_counts())
print(f"\nOverall churn rate: {df_churn['Churn'].mean() * 100:.1f}%")

Churn column distribution:

Churn
0    5174
1    1869
Name: count, dtype: int64

Overall churn rate: 26.5%


## 9. Churn Reasons
We analyse why churned customers left the company, breaking down churn by category to uncover the key business drivers behind customer attrition.

In [10]:
# look at churn reasons — only makes sense to look at churned customers
# so we filter first, then count
churned_only = df_churn[df_churn['Churn'] == 1]

print("Churn Category breakdown:\n")
cat_counts = churned_only['Churn Category'].value_counts()
cat_pct    = (cat_counts / len(churned_only) * 100).round(1)

for category, count in cat_counts.items():
    print(f"  {category:<20}  {count:>4}  ({cat_pct[category]}%)")

Churn Category breakdown:

  Competitor             841  (45.0%)
  Dissatisfaction        321  (17.2%)
  Attitude               314  (16.8%)
  Price                  211  (11.3%)
  Other                  182  (9.7%)


## 10. Numeric Columns
We review the statistical summary of key numeric columns to understand their range, average, and spread, and flag any anomalies worth investigating.

In [11]:
# descriptive statistics for key numeric columns
# these are the columns most likely to influence churn
key_numeric = [
    'Age',
    'Tenure in Months',
    'Monthly Charge',
    'Total Charges',
    'Total Refunds',
    'Total Revenue',
    'Number of Referrals'
]

# .describe() gives count, mean, std, min, quartiles and max
df_churn[key_numeric].describe().round(2)

,Age,Tenure in Months,Monthly Charge,Total Charges,Total Refunds,Total Revenue,Number of Referrals
count,7043.00,7043.00,7043.00,7043.00,7043.00,7043.00,7043.00
mean,46.51,32.39,63.60,2280.38,1.96,3034.38,1.95
std,16.75,24.54,31.20,2266.22,7.90,2865.20,3.00
min,19.00,1.00,-10.00,18.80,0.00,21.36,0.00
25%,32.00,9.00,30.40,400.15,0.00,605.61,0.00
50%,46.00,29.00,70.05,1394.55,0.00,2108.64,0.00
75%,60.00,55.00,89.75,3786.60,0.00,4801.15,3.00
max,80.00,72.00,118.75,8684.80,49.79,11979.34,11.00


## Key Numeric Column Findings

- Average tenure is 32 months (~2.5 years)
- Monthly charges range from -$10 (credit applied) to $118.75
- 75% of customers have zero refunds
- Referrals range from 0 to 11 — loyal customers actively refer others
- Negative Monthly Charge flagged as anomaly — likely a billing credit

## 11. Categorical Columns
We inspect the value counts of key categorical columns to understand their distribution and confirm there are no unexpected or inconsistent values.

In [12]:
# check value counts for key categorical columns
# these are the ones most relevant to churn analysis
cat_cols = [
    'Contract',
    'Payment Method',
    'Internet Type',
    'Offer',
    'Gender',
    'Married'
]

for col in cat_cols:
    counts = df_churn[col].value_counts(dropna=False)
    pct    = (counts / len(df_churn) * 100).round(1)
    print(f"\n{col}:")
    for val, count in counts.items():
        print(f"  {str(val):<30}  {count:>5,}  ({pct[val]}%)")


Contract:
  Month-to-Month                  3,610  (51.3%)
  Two Year                        1,883  (26.7%)
  One Year                        1,550  (22.0%)

Payment Method:
  Bank Withdrawal                 3,909  (55.5%)
  Credit Card                     2,749  (39.0%)
  Mailed Check                      385  (5.5%)

Internet Type:
  Fiber Optic                     3,035  (43.1%)
  DSL                             1,652  (23.5%)
  nan                             1,526  (21.7%)
  Cable                             830  (11.8%)

Offer:
  nan                             3,877  (55.0%)
  Offer B                           824  (11.7%)
  Offer E                           805  (11.4%)
  Offer D                           602  (8.5%)
  Offer A                           520  (7.4%)
  Offer C                           415  (5.9%)

Gender:
  Male                            3,555  (50.5%)
  Female                          3,488  (49.5%)

Married:
  No                              3,641  (51.7%)
  

## Key Categorical Column Findings

- 51.3% of customers are on Month-to-Month contracts — highest churn risk group
- 55% of customers have no marketing offer applied
- Fiber Optic is the most common internet type (43.1%)
- Gender and marital status are nearly 50/50 — unlikely to be strong predictors
- 1,526 nulls in Internet Type confirmed as no-internet customers

## 12. Zip Code Join Validation

We validate that all zip codes in the churn table exist in the population table, ensuring no customers will lose their data when the two tables are joined in PostgreSQL.

In [ ]:
# get unique zip codes from both tables
churn_zips = set(df_churn['Zip Code'].unique())
pop_zips   = set(df_zip['Zip Code'].unique())

# find matches and mismatches
matched   = churn_zips & pop_zips   # & = intersection (in both)
unmatched = churn_zips - pop_zips   # - = difference (in churn but not in pop)

print(f"Unique zip codes in churn table:      {len(churn_zips):,}")
print(f"Unique zip codes in population table: {len(pop_zips):,}")
print(f"Matched:                              {len(matched):,}")
print(f"Unmatched:                            {len(unmatched):,}")

if len(unmatched) == 0:
    print("\n✓ All zip codes match. JOIN is safe.")
else:
    print(f"\n⚠ Warning: {len(unmatched)} zip codes won't match in a JOIN.")
    

Unique zip codes in churn table:      1,626
Unique zip codes in population table: 1,671
Matched:                              1,626
Unmatched:                            0

✓ All zip codes match. JOIN is safe.


## Notebook Summary

| Check | Result |
|---|---|
| Rows loaded | 7,043 customers × 38 columns |
| Missing values | 15 columns — all structural, not errors |
| Target variable | 26.5% churn rate — mild class imbalance |
| Top churn reason | Competitor (45% of churned customers) |
| Highest risk group | Month-to-Month contract customers (51.3%) |
| Zip code join | 100% coverage — JOIN is safe |
| Anomaly flagged | Negative Monthly Charge (-$10) — likely credit |

## Next Step
`02_sql_analysis.ipynb` — load both tables into PostgreSQL,
design the schema, and write analytical SQL queries.